In [ ]:
#!pip install -q transformers datasets sacrebleu sentencepiece
!pip uninstall -y transformers
!pip install -q "transformers==4.41.2" "sentencepiece" "sacrebleu"

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2


In [ ]:
import transformers
print(transformers.__version__)

4.41.2


In [ ]:
import torch
import time
import json
import numpy as np
import random

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [ ]:
# from google.colab import files
# files.upload()  # upload kaggle.json

In [ ]:
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# !kaggle datasets download -d mathurinache/flores101
# !unzip -q flores101.zip -d flores101
# !ls flores101

In [ ]:
import os

base_path = "/content/flores101/flores101_dataset/devtest"

SRC = "eng"   # source language file code
TGT = "zul"   # target language file code

src_file = os.path.join(base_path, f"{SRC}.devtest")
tgt_file = os.path.join(base_path, f"{TGT}.devtest")

def load_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [l.strip() for l in f]

sources = load_lines(src_file)
references = load_lines(tgt_file)

assert len(sources) == len(references), "Parallel files misaligned!"

print("Samples:", len(sources))
print("SRC:", sources[0])
print("REF:", references[0])

Samples: 1012
SRC: "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.
REF: "Manje sinamagundane anezinyanga ezingu-4 angenawo ushukela ayenoshukela," wanezela.


In [ ]:
# def load_flores_file(path):
#     with open(path, "r", encoding="utf-8") as f:
#         lines = [line.strip() for line in f.readlines()]
#     return lines

# sources = load_flores_file(src_file)
# references = load_flores_file(tgt_file)

# print("Samples:", len(sources))
# print(sources[0])
# print(references[0])

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model_name = "AfriNLP/AfriNLLB-12enc-12dec-full-ft-kd"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.to(device)
model.eval()

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [ ]:
import time

def translate(texts, src_lang="eng_Latn", tgt_lang="zul_Latn", batch_size=8, max_len=200):
    tokenizer.src_lang = src_lang
    outputs = []

    start = time.time()

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            gen = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang],
                max_length=max_len
            )

        decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        outputs.extend(decoded)

    total = time.time() - start
    latency = total / len(texts)
    throughput = len(texts) / total

    return outputs, latency, throughput

In [ ]:
predictions, latency, throughput = translate(
    sources,
    src_lang="eng_Latn",
    tgt_lang="zul_Latn",
    batch_size=8
)

print("Latency (sec/sent):", latency)
print("Throughput (sent/sec):", throughput)
print("Example MT:", predictions[0])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
the `lang_code_to_id` attribute is deprecated. The logic is natively handled in the `tokenizer.adder_tokens_decoder` this attribute will be removed in `transformers` v4.38
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1283: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


Latency (sec/sent): 0.6306344362586854
Throughput (sent/sec): 1.5857047165591212
Example MT: "Manje sinezimbuzi ezinezinyanga ezine ubudala ezingenawo ushukela ezazikade zinoshukela", enezela.


In [ ]:
import sacrebleu

chrf = sacrebleu.corpus_chrf(predictions, [references])
print("chrF++:", chrf.score)

chrF++: 57.156174645697334


In [ ]:
!pip install --upgrade pip
!pip install unbabel-comet

In [ ]:
import json

results = {
    "model": model_name,
    "source_lang": "eng_Latn",
    "target_lang": "zul_Latn",
    "chrF++": chrf.score,
    "latency": latency,
    "throughput": throughput,
    "num_samples": len(sources)
}

with open("baseline_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

with open("baseline_predictions.json", "w") as f:
    json.dump(predictions, f, indent=2)

print("Saved baseline files.")

Saved baseline files.


# COMET Evaluation

In [ ]:
import json

# Load predictions
with open("baseline_predictions.json") as f:
    predictions = json.load(f)

# Reload FLORES data (same way as before)
def load_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [l.strip() for l in f]

base_path = "/content/flores101/flores101_dataset/devtest"

sources = load_lines(f"{base_path}/eng.devtest")
references = load_lines(f"{base_path}/zul.devtest")

In [ ]:
# Install COMET (run once in this notebook)
# !pip install -q "unbabel-comet==2.2.2"

from comet import download_model, load_from_checkpoint
import json
import os

# 🔹 1. Load AfriCOMET model via COMET checkpoint
try:
    model_path = download_model("masakhane/africomet")
except:
    print("Primary AfriCOMET checkpoint not found, trying fallback...")
    model_path = download_model("masakhane/africomet-qe-stl")

model = load_from_checkpoint(model_path)

# 🔹 2. Prepare evaluation data
data = [
    {"src": s, "mt": p, "ref": r}
    for s, p, r in zip(sources, predictions, references)
]

# 🔹 3. Run evaluation
result = model.predict(
    data,
    batch_size=8,
    gpus=1 if device == "cuda" else 0
)

africomet_score = result["system_score"]
print("AfriCOMET:", africomet_score)

# 🔹 4. Load existing baseline metrics (if exists)
metrics_file = "baseline_metrics.json"

if os.path.exists(metrics_file):
    with open(metrics_file, "r") as f:
        metrics = json.load(f)
else:
    metrics = {}

# 🔹 5. Append AfriCOMET score
metrics["AfriCOMET"] = africomet_score

# 🔹 6. Save updated metrics
with open(metrics_file, "w") as f:
    json.dump(metrics, f, indent=2)

print("Updated baseline_metrics.json with AfriCOMET score.")

Primary AfriCOMET checkpoint not found, trying fallback...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

hparams.yaml:   0%|          | 0.00/573 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/714 [00:00<?, ?B/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|

AfriCOMET: 0.7383714636139895
Updated baseline_metrics.json with AfriCOMET score.
